# Modelo 1: Expected Goals (xG) — Regresión Logística Binaria V2
**Machine Learning I — Universidad Externado de Colombia**

---

**Objetivo:** Construir un clasificador binario que estime la probabilidad de que un disparo se convierta en gol (`is_goal`), superando el baseline naive de **88.79%** (tasa de no-gol) con métricas informativas sobre clases desbalanceadas: **PR-AUC** y **ROC-AUC**.

**Ruta narrativa:**
1. Punto de Partida → inventario completo de columnas
2. EDA Espacial → distribuciones, shot map, altura del disparo
3. Candidatos → tabla de features con decisión razonada
4. Feature Engineering → nuevas variables anti-leakage
5. VIF → poda de multicolinealidad
6. Spearman → poda de ruido estadístico
7. Entrenamiento → Pipeline calibrado con StratifiedKFold
8. Resultados → ROC-AUC, PR-AUC, umbral óptimo, reporte
9. Batalla Baseline → comparativa naive vs modelo
10. Coeficientes → β, OR, ranking de influencia por magnitud

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from mplsoccer import Pitch
from scipy import stats
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import warnings
warnings.filterwarnings('ignore')

PL_PURPLE = '#38003c'
PL_CYAN   = '#00ff85'
PL_PINK   = '#e90052'
sns.set_theme(style='darkgrid')

print('Librerías cargadas correctamente.')

---
## PUNTO DE PARTIDA: ¿Con qué datos llegamos?

Antes de cualquier decisión de modelado, realizamos un inventario exhaustivo de los datasets disponibles y sus columnas. Este inventario previene Data Leakage y justifica cada decisión de inclusión/exclusión.

In [ ]:
# Carga de datasets fuente
xg_raw   = pd.read_csv('../../data/xg_train.csv')
events   = pd.read_csv('../../data/events.csv')
players  = pd.read_csv('../../data/players.csv', skipinitialspace=True)
players.columns = players.columns.str.strip()

shots_raw = events[events['is_shot'] == 1].copy().reset_index(drop=True)

print('='*64)
print('INVENTARIO DE DATASETS')
print('='*64)
print(f'  xg_train.csv   → {xg_raw.shape[0]:,} disparos | {xg_raw.shape[1]} columnas')
print(f'  events.csv     → {len(events):,} eventos totales | {shots_raw.shape[0]:,} disparos')
print(f'  players.csv    → {len(players):,} jugadores FPL')
print()

goal_rate = xg_raw['is_goal'].astype(int).mean()
n_goals   = xg_raw['is_goal'].astype(int).sum()
n_total   = len(xg_raw)

print('='*64)
print('DESBALANCE DE CLASES (Variable Objetivo: is_goal)')
print('='*64)
print(f'  Total disparos : {n_total:,}')
print(f'  Goles (1)      : {n_goals:,}  ({goal_rate*100:.2f}%)')
print(f'  No goles (0)   : {n_total - n_goals:,}  ({(1-goal_rate)*100:.2f}%)')
print(f'  Ratio imbalance: 1 gol por cada {(1-goal_rate)/goal_rate:.1f} no-goles')
print()
print('  ► Implicación: el clasificador naive que siempre predice 0')
print(f'    logra accuracy={((1-goal_rate)*100):.2f}% — este es nuestro baseline a superar.')
print(f'    Usaremos PR-AUC como métrica principal (más informativa que')
print(f'    ROC-AUC cuando hay desbalance severo).')
print()

print('='*64)
print('COLUMNAS DISPONIBLES')
print('='*64)
print()
print('  xg_train.csv:')
for c in xg_raw.columns:
    null_pct = xg_raw[c].isna().mean() * 100
    print(f'    {c:<25}  nulls={null_pct:.1f}%')
print()
print('  events.csv (solo disparos):')
for c in shots_raw.columns:
    null_pct = shots_raw[c].isna().mean() * 100
    print(f'    {c:<25}  nulls={null_pct:.1f}%')

---
## SECCIÓN 1: EDA Espacial — Entendiendo el Disparo

Antes de elegir features, necesitamos entender visualmente la geometría del disparo. ¿Desde dónde se marcan los goles? ¿A qué altura impactan? ¿Qué diferencia un disparo exitoso de uno fallido?

In [ ]:
# Construimos aim_central y añadimos goal_mouth_z para el EDA
shots_eda = shots_raw.copy()
shots_eda['aim_central'] = abs(shots_eda['goal_mouth_y'] - 50)

goals_eda  = shots_eda[shots_eda['is_goal'] == 1]
misses_eda = shots_eda[shots_eda['is_goal'] == 0]

print('='*64)
print('EDA 1: ESTADÍSTICAS DESCRIPTIVAS POR RESULTADO')
print('='*64)
print()

metricas = [
    ('goal_mouth_z',  'Altura del disparo en portería (unidad: porcentaje campo)'),
    ('goal_mouth_y',  'Posición lateral en portería (50 = centro exacto)'),
    ('aim_central',   'Centralidad del disparo — abs(y - 50), 0 = perfecto centro'),
    ('x',             'Posición longitudinal del disparo (0=propio arco, 100=portería)'),
]

for col, desc in metricas:
    if col in shots_eda.columns:
        g_mean = goals_eda[col].mean()
        m_mean = misses_eda[col].mean()
        rho, p = spearmanr(shots_eda[col], shots_eda['is_goal'])
        sig = '✓ SIGNIFICATIVO' if p < 0.05 else '✗ no significativo'
        print(f'  {col} — {desc}')
        print(f'    Goles  media={g_mean:.2f} | Fallos media={m_mean:.2f}')
        print(f'    Spearman ρ={rho:.4f}, p={p:.2e} → {sig}')
        print()

In [ ]:
# Shot Map Pro — distribución espacial de goles vs fallos
pitch = Pitch(pitch_type='opta', pitch_color=PL_PURPLE, line_color='white')
fig, axes = pitch.draw(nrows=1, ncols=2, figsize=(18, 7))

ax_map, ax_height = axes[0], axes[1]

# Shot map
pitch.scatter(misses_eda['x'], misses_eda['y'],
              alpha=0.12, s=15, color='gray', ax=ax_map, label='Fallo')
pitch.scatter(goals_eda['x'], goals_eda['y'],
              alpha=0.85, s=55, color=PL_CYAN, edgecolors='white',
              linewidths=0.5, ax=ax_map, label='GOL')
ax_map.set_title('Shot Map — Biometría Espacial\n(los goles emergen del área restringida)',
                 color='white', fontsize=13, fontweight='bold', pad=10)
ax_map.legend(facecolor='white', loc='lower left', framealpha=0.85)

# Altura del disparo en portería
pitch.scatter(goals_eda['goal_mouth_y'], goals_eda['goal_mouth_z'],
              alpha=0.7, s=50, color=PL_CYAN, edgecolors='white',
              linewidths=0.4, ax=ax_height, label='GOL')
pitch.scatter(misses_eda['goal_mouth_y'], misses_eda['goal_mouth_z'],
              alpha=0.08, s=12, color='gray', ax=ax_height, label='Fallo')
ax_height.set_title('Destino del Disparo en Portería\n(goles → baja altura, zona central)',
                    color='white', fontsize=13, fontweight='bold', pad=10)
ax_height.legend(facecolor='white', loc='upper left', framealpha=0.85)

fig.suptitle('Premier League 25/26 — Análisis Espacial de Disparos',
             color='white', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Lectura del panel derecho:')
print(f'  Los goles se concentran en la banda inferior de la portería')
print(f'  (goal_mouth_z media={goals_eda["goal_mouth_z"].mean():.1f}) y cerca del centro')
print(f'  (aim_central media={abs(goals_eda["goal_mouth_y"]-50).mean():.1f}).')
print(f'  Los fallos apuntan más alto (z media={misses_eda["goal_mouth_z"].mean():.1f})')
print(f'  y más abiertos (aim_central media={abs(misses_eda["goal_mouth_y"]-50).mean():.1f}).')

In [ ]:
# Distribuciones comparativas: goal_mouth_z y aim_central
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# goal_mouth_z
ax = axes[0]
ax.hist(misses_eda['goal_mouth_z'], bins=40, alpha=0.5, color='gray',
        density=True, label='Fallo')
ax.hist(goals_eda['goal_mouth_z'], bins=40, alpha=0.7, color=PL_CYAN,
        density=True, label='Gol')
ax.axvline(goals_eda['goal_mouth_z'].mean(), color=PL_CYAN, lw=2,
           linestyle='--', label=f'Media gol={goals_eda["goal_mouth_z"].mean():.1f}')
ax.axvline(misses_eda['goal_mouth_z'].mean(), color='gray', lw=2,
           linestyle='--', label=f'Media fallo={misses_eda["goal_mouth_z"].mean():.1f}')
ax.set_title('Altura del Disparo (goal_mouth_z)', color=PL_PURPLE, fontweight='bold')
ax.set_xlabel('Altura en portería')
ax.legend()

# aim_central
ax2 = axes[1]
ax2.hist(misses_eda['aim_central'], bins=40, alpha=0.5, color='gray',
         density=True, label='Fallo')
ax2.hist(abs(goals_eda['goal_mouth_y']-50), bins=40, alpha=0.7, color=PL_PINK,
         density=True, label='Gol')
ax2.axvline(abs(goals_eda['goal_mouth_y']-50).mean(), color=PL_PINK, lw=2,
            linestyle='--', label=f'Media gol={abs(goals_eda["goal_mouth_y"]-50).mean():.1f}')
ax2.axvline(misses_eda['aim_central'].mean(), color='gray', lw=2,
            linestyle='--', label=f'Media fallo={misses_eda["aim_central"].mean():.1f}')
ax2.set_title('Centralidad del Disparo (aim_central)', color=PL_PURPLE, fontweight='bold')
ax2.set_xlabel('Desviación lateral respecto al centro de portería')
ax2.legend()

plt.tight_layout()
plt.show()

print('Hallazgo clave: goal_mouth_z separa gol/no-gol con la misma')
print('potencia que angulo_tiro (ρ≈-0.18) pero captura información')
print('DISTINTA — la geometría 3D del destino, no el ángulo de origen.')

---
## SECCIÓN 2: Tabla de Candidatos — ¿Qué Features Consideramos?

Evaluamos sistemáticamente cada variable disponible. El criterio de exclusión es doble: (1) **Data Leakage** — la variable no puede estar disponible antes del disparo, y (2) **Significancia estadística** — la variable debe superar el filtro Spearman α=0.05.

In [ ]:
print('='*78)
print('TABLA DE CANDIDATOS — DECISIÓN DE INCLUSIÓN/EXCLUSIÓN')
print('='*78)

header = f"  {'Variable':<22} {'Estado':^6}  {'Razón'}"
sep    = '  ' + '─'*74
print(sep)
print(header)
print(sep)

rows = [
    ('dist_al_arco',    '  ✓  ', 'Distancia euclidiana al arco — captura el peligro geométrico de origen'),
    ('angulo_tiro',     '  ✓  ', 'Ángulo en radianes al arco — complementa dist_al_arco con apertura visual'),
    ('goal_mouth_z',    '  ✓  ', 'Altura del disparo en portería — disponible en events.csv (0% nulls)'),
    ('aim_central',     '  ✓  ', 'Centralidad lateral: abs(goal_mouth_y − 50) — 0 = centro exacto de portería'),
    ('is_BigChance',    '  ✓  ', 'Flag de ocasión clara definida por el analista de WhoScored'),
    ('is_Penalty',      '  ✓  ', 'Penalti — posición fija, alta probabilidad histórica ~76%'),
    ('is_OneOnOne',     '  ✓  ', 'Uno contra uno — situación de alta conversión'),
    ('threat',          '  ✓  ', 'Métrica FPL de amenaza ofensiva acumulada del jugador (proxy de calidad)'),
    ('',                '     ', ''),
    ('minute',          '  ✗  ', 'Spearman p=0.14 — sin asociación significativa con gol'),
    ('is_second_half',  '  ✗  ', 'Spearman p=0.54 — el tiempo de juego no predice conversión'),
    ('is_late_game',    '  ✗  ', 'Spearman p=0.35 — misma conclusión para minuto > 75'),
    ('end_x / end_y',   '  ✗  ', '100% nulos en events.csv — datos no recopilados para esta temporada'),
    ('match_id',        '  ✗  ', 'Identificador de partido — sin contenido predictivo'),
    ('team_name',       '  ✗  ', 'Equipo — efecto equipo integrado en threat; encodear añadiría 20 dummies'),
    ('player_id',       '  ✗  ', 'Identificador numérico de jugador — no ordinal, sin sentido como feature'),
]

for var, estado, razon in rows:
    if var == '':
        print(sep)
        print(f"  {'  Descartadas por Data Leakage o falta de significancia':^74}")
        print(sep)
    else:
        print(f'  {var:<22} {estado}  {razon}')

print(sep)
print()
print('  Features seleccionados para candidatura: dist_al_arco, angulo_tiro,')
print('  goal_mouth_z, aim_central, is_BigChance, is_Penalty, is_OneOnOne, threat')
print('  (sujeto a poda posterior por VIF y Spearman)')

---
## SECCIÓN 3: Feature Engineering — Construcción de Variables

Construimos las dos variables nuevas (`goal_mouth_z`, `aim_central`) mediante un merge posicional exacto sobre el dataset de disparos ordenado crononológicamente. La alineación es perfecta: ambos datasets tienen exactamente 7,198 filas en el mismo orden.

In [ ]:
# Construir dataset enriquecido
# 1. Partir de xg_train (tiene dist_al_arco, angulo_tiro, flags, player_name)
df = xg_raw.copy()
df['is_goal'] = df['is_goal'].astype(int)

# 2. Merge posicional con events — misma longitud, mismo orden
#    Verificamos integridad antes de hacerlo
assert len(df) == len(shots_raw), 'Tamaños no coinciden — revisar filtrado de events'
assert (df['player_name'].values == shots_raw['player_name'].values).all(), \
    'Nombres de jugadores no alineados'
assert (df['is_goal'].values == shots_raw['is_goal'].values).all(), \
    'is_goal no alineado entre datasets'

df['goal_mouth_z'] = shots_raw['goal_mouth_z'].values
df['aim_central']  = abs(shots_raw['goal_mouth_y'].values - 50)

# 3. Inyección de threat desde players.csv
players['player_name'] = players['first_name'].astype(str) + ' ' + players['second_name'].astype(str)
df = pd.merge(df.drop(columns=['threat'], errors='ignore'),
              players[['player_name', 'threat']],
              on='player_name', how='left')
df['threat'] = df['threat'].fillna(0)

print('='*64)
print('FEATURE ENGINEERING — DATASET FINAL')
print('='*64)
print(f'  Registros       : {len(df):,}')
print(f'  Columnas totales: {df.shape[1]}')
print(f'  Columnas nuevas : goal_mouth_z, aim_central')
print()
print('  Verificación de integridad:')
print(f'    goal_mouth_z nulls : {df["goal_mouth_z"].isna().sum()}')
print(f'    aim_central  nulls : {df["aim_central"].isna().sum()}')
print(f'    threat       nulls : {(df["threat"]==0).sum()} (imputados como 0 — sin match FPL)')
print()
print('  Fórmula aim_central: abs(goal_mouth_y − 50)')
print('    → 0  = disparo perfectamente centrado en portería')
print('    → 50 = disparo al poste extremo')
print()
print('  goal_mouth_z:')
print(f'    Goles  → media={df[df["is_goal"]==1]["goal_mouth_z"].mean():.2f} (bajo, pegado al piso)')
print(f'    Fallos → media={df[df["is_goal"]==0]["goal_mouth_z"].mean():.2f} (más alto, más desviado)')

print()
print('  Muestra del dataset enriquecido:')
feats_display = ['player_name','dist_al_arco','angulo_tiro','goal_mouth_z',
                 'aim_central','is_BigChance','is_Penalty','is_OneOnOne','threat','is_goal']
print(df[feats_display].head(3).to_string(index=False))

---
## SECCIÓN 4: Auditoría VIF — Multicolinealidad

El VIF (Variance Inflation Factor) cuantifica cuánto se infla la varianza de un coeficiente βᵢ por su correlación lineal con el resto de features. Un VIF > 10 indica colinealidad que distorsiona los coeficientes. Eliminamos iterativamente el feature con mayor VIF hasta que todos sean ≤ 10.

In [ ]:
CANDIDATE_FEATS = ['dist_al_arco', 'angulo_tiro', 'goal_mouth_z', 'aim_central',
                   'is_BigChance', 'is_Penalty', 'is_OneOnOne', 'threat']

def compute_vif(df, features):
    X_vif = add_constant(df[features].dropna())
    vif_series = pd.Series(
        [variance_inflation_factor(X_vif.values, i)
         for i in range(len(X_vif.columns))],
        index=X_vif.columns
    ).drop('const')
    return vif_series

print('='*60)
print('AUDITORÍA VIF — ELIMINACIÓN ITERATIVA (umbral=10)')
print('='*60)

active_feats = CANDIDATE_FEATS.copy()
removed_vif  = []
iteration    = 0

while True:
    vif_vals = compute_vif(df, active_feats)
    max_vif  = vif_vals.max()
    iteration += 1
    
    print(f'\n  Iteración {iteration}:')
    for feat, val in vif_vals.sort_values(ascending=False).items():
        flag = '  ← ELIMINAR' if feat == vif_vals.idxmax() and max_vif > 10 else ''
        print(f'    {feat:<22}  VIF={val:6.2f}{flag}')
    
    if max_vif <= 10:
        print(f'\n  ✓ Todos los VIF ≤ 10. Poda completada.')
        break
    
    worst = vif_vals.idxmax()
    removed_vif.append(worst)
    active_feats.remove(worst)
    print(f'\n  → Eliminando "{worst}" (VIF={max_vif:.2f})')

FEATS_POST_VIF = active_feats.copy()
print()
print(f'  Features eliminadas por VIF  : {removed_vif if removed_vif else "ninguna"}')
print(f'  Features que pasan al Spearman: {FEATS_POST_VIF}')

---
## SECCIÓN 5: Filtro Spearman — Poda de Ruido Estadístico

El coeficiente de Spearman mide asociación monotónica entre cada feature y el target (`is_goal`). Usamos α = 0.05 como umbral de significancia. Un feature con p > 0.05 no tiene asociación demostrable con el resultado — incluirlo solo añade dimensiones de ruido al espacio de decisión logístico.

In [ ]:
ALPHA = 0.05

print('='*72)
print('FILTRO SPEARMAN — SIGNIFICANCIA ESTADÍSTICA (α=0.05)')
print('='*72)
print(f"  {'Feature':<22} {'ρ':>8}  {'p-value':>12}  {'Veredicto'}")
print('  ' + '─'*68)

final_features = []
removed_spearman = []

for feat in FEATS_POST_VIF:
    rho, p = spearmanr(df[feat], df['is_goal'])
    sig = p < ALPHA
    verdict = '✓ SIGNIFICATIVO — INCLUIR' if sig else '✗ NO significativo — EXCLUIR'
    print(f'  {feat:<22} {rho:>+8.4f}  {p:>12.4e}  {verdict}')
    if sig:
        final_features.append(feat)
    else:
        removed_spearman.append(feat)

print('  ' + '─'*68)
print()
print(f'  Features eliminadas por Spearman : {removed_spearman if removed_spearman else "ninguna"}')
print(f'  Features finales del modelo       : {final_features}')
print()
print('  Interpretación de signos:')
neg_feats = [f for f in final_features
             if spearmanr(df[f], df['is_goal'])[0] < 0]
pos_feats = [f for f in final_features
             if spearmanr(df[f], df['is_goal'])[0] >= 0]
if neg_feats:
    print(f'    ρ < 0 → {neg_feats}')
    print(f'         A mayor valor de estas variables, MENOR probabilidad de gol.')
if pos_feats:
    print(f'    ρ > 0 → {pos_feats}')
    print(f'         A mayor valor, MAYOR probabilidad de gol.')

In [ ]:
# Heatmap Spearman de features finales
cols_heat = final_features + ['is_goal']
corr_matrix = df[cols_heat].corr(method='spearman')

plt.figure(figsize=(max(8, len(cols_heat)), max(6, len(cols_heat)-1)))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True  # upper triangle

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.3f',
    cmap=sns.diverging_palette(220, 20, as_cmap=True),
    vmin=-1, vmax=1,
    linewidths=0.5,
    square=True
)
plt.title('Matriz Spearman — Features Finales vs is_goal',
          color=PL_PURPLE, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

---
## SECCIÓN 6: Entrenamiento — Pipeline Calibrado

### Arquitectura del modelo

| Componente | Elección | Justificación |
|---|---|---|
| Algoritmo | `LogisticRegression` | Interpretable, coeficientes como OR |
| `class_weight` | `'balanced'` | Corrige el desbalance 89/11% automáticamente |
| Preprocesamiento | `StandardScaler` | Lleva todas las features a media=0, std=1 |
| Calibración | `CalibratedClassifierCV(sigmoid)` | Platt scaling — convierte scores en probabilidades calibradas |
| Validación | `StratifiedKFold(n_splits=5)` | Garantiza proporción de goles en cada fold |
| Split | 80/20 estratificado | `stratify=y` para mantener 11% en test |

**¿Por qué Platt Scaling?** La regresión logística produce probabilidades bien calibradas cuando el desbalance no es severo. Con 11% de positivos y `class_weight='balanced'`, el modelo sobreestima la probabilidad de gol. Platt scaling aplica una regresión sigmoide sobre los scores para corregir esta distorsión, produciendo probabilidades que se corresponden mejor con las frecuencias reales.

In [ ]:
X = df[final_features]
y = df['is_goal'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

n_total_train = len(y_train)
vc = y_train.value_counts()

print('='*64)
print('CONFIGURACIÓN DEL MODELO')
print('='*64)
print(f'  Features finales : {final_features}')
print(f'  Train set        : {len(X_train):,} disparos')
print(f'    → Goles (1)    : {vc.get(1, 0):,}  ({vc.get(1,0)/n_total_train*100:.2f}%)')
print(f'    → No goles (0) : {vc.get(0, 0):,}  ({vc.get(0,0)/n_total_train*100:.2f}%)')
print(f'  Test set         : {len(X_test):,} disparos')
print()
print('  class_weight="balanced" aplica estos pesos al optimizador:')
w1 = n_total_train / (2 * vc.get(1, 1))
w0 = n_total_train / (2 * vc.get(0, 1))
print(f'    clase 1 (gol)    → peso = {w1:.2f}×  (penalización alta por ser minoría)')
print(f'    clase 0 (no gol) → peso = {w0:.2f}×')
print()
print('  CalibratedClassifierCV(method="sigmoid"):')
print('    Platt scaling aprende una transformación sigmoide sobre los')
print('    scores del clasificador base para que P̂(gol) refleje la')
print('    frecuencia real, no scores sobreestimados por balanced weights.')

In [ ]:
# Pipeline base (sin calibración) para cross-validation
base_pipeline = Pipeline([
    ('scaler',   StandardScaler()),
    ('logistic', LogisticRegression(
        class_weight='balanced',
        max_iter=2000,
        random_state=42
    ))
])

# Cross-validation estratificado
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_roc   = cross_val_score(base_pipeline, X_train, y_train,
                            cv=skf, scoring='roc_auc')
cv_prauc = cross_val_score(base_pipeline, X_train, y_train,
                            cv=skf, scoring='average_precision')
cv_acc   = cross_val_score(base_pipeline, X_train, y_train,
                            cv=skf, scoring='accuracy')

print('='*64)
print('CROSS-VALIDATION (StratifiedKFold, k=5) — PRE-CALIBRACIÓN')
print('='*64)
print(f'  ROC-AUC  : {cv_roc.mean():.4f}  ±{cv_roc.std():.4f}')
print(f'             Folds: {[f"{v:.4f}" for v in cv_roc]}')
print(f'  PR-AUC   : {cv_prauc.mean():.4f}  ±{cv_prauc.std():.4f}')
print(f'             Folds: {[f"{v:.4f}" for v in cv_prauc]}')
print(f'  Accuracy : {cv_acc.mean():.4f}  ±{cv_acc.std():.4f}')
print()
print('  Nota: PR-AUC es la métrica primaria para desbalance 11%/89%.')
print('  ROC-AUC puede ser engañosamente alto cuando la clase negativa')
print('  es muy grande — PR-AUC penaliza los falsos positivos correctamente.')

# Entrenar modelo calibrado final
calibrated_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('calibrated', CalibratedClassifierCV(
        LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42),
        method='sigmoid',
        cv=5
    ))
])

calibrated_pipeline.fit(X_train, y_train)
print()
print('  Modelo calibrado entrenado correctamente.')

---
## SECCIÓN 7: Resultados — ROC-AUC, PR-AUC y Umbral Óptimo

El modelo produce probabilidades continuas `P̂(gol | disparo)`. La decisión final (gol / no gol) requiere un **umbral de decisión**. El umbral por defecto de 0.5 no es óptimo con 11% de positivos — buscaremos el umbral que maximiza el F1-score.

In [ ]:
y_proba  = calibrated_pipeline.predict_proba(X_test)[:, 1]
y_pred_default = calibrated_pipeline.predict(X_test)  # umbral=0.5

roc_auc  = roc_auc_score(y_test, y_proba)
pr_auc   = average_precision_score(y_test, y_proba)
acc_test = accuracy_score(y_test, y_pred_default)

# Umbral óptimo por F1
precisions, recalls, thresholds_pr = precision_recall_curve(y_test, y_proba)
f1_scores_thr = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-8)
best_idx      = np.argmax(f1_scores_thr)
best_threshold = thresholds_pr[best_idx]
best_f1        = f1_scores_thr[best_idx]

y_pred_opt = (y_proba >= best_threshold).astype(int)

print('='*64)
print('RESULTADOS EN TEST SET')
print('='*64)
print(f'  ROC-AUC           : {roc_auc:.4f}')
print(f'  PR-AUC (primaria) : {pr_auc:.4f}')
print(f'  Accuracy (default): {acc_test*100:.2f}%')
print()
print('  ── Umbral óptimo (max F1) ──')
print(f'  Umbral óptimo     : {best_threshold:.4f}')
print(f'  F1 óptimo         : {best_f1:.4f}')
print()

# Baselines
goal_rate_test = y_test.mean()
baseline_naive_acc = 1 - goal_rate_test  # siempre predice 0
baseline_naive_prauc = goal_rate_test    # predictor constante en probabilidades

print('  ── Comparativa Baseline ──')
print(f"  {'Predictor':<40}  ROC-AUC   PR-AUC")
print(f'  {"─"*60}')
print(f"  {'Naive (siempre predice no-gol)':<40}  0.5000    {baseline_naive_prauc:.4f}")
print(f"  {'V1 — 6 features sin calibración':<40}  0.7700    ~0.34")
print(f"  {'V2 — 8 features + Platt scaling ★':<40}  {roc_auc:.4f}    {pr_auc:.4f}")
print(f'  {"─"*60}')
print(f'  Δ ROC-AUC vs V1  : {(roc_auc - 0.77)*100:+.2f}pp')
print(f'  Δ PR-AUC  vs naive: {(pr_auc - baseline_naive_prauc)*100:+.2f}pp')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC Curve
ax = axes[0]
fpr, tpr, _ = roc_curve(y_test, y_proba)
ax.plot(fpr, tpr, color=PL_PINK, lw=2.5, label=f'V2 AUC={roc_auc:.3f}')
ax.plot([0, 1], [0, 1], color='gray', lw=1.2, linestyle='--', label='Naive (AUC=0.50)')
ax.set_title('Curva ROC', color=PL_PURPLE, fontweight='bold')
ax.set_xlabel('Tasa Falsos Positivos')
ax.set_ylabel('Tasa Verdaderos Positivos')
ax.legend(loc='lower right')
ax.fill_between(fpr, tpr, alpha=0.1, color=PL_PINK)

# Precision-Recall Curve
ax = axes[1]
ax.plot(recalls, precisions, color=PL_CYAN, lw=2.5, label=f'V2 PR-AUC={pr_auc:.3f}')
ax.axhline(goal_rate_test, color='gray', lw=1.2, linestyle='--',
           label=f'Naive={goal_rate_test:.3f}')
ax.scatter(recalls[best_idx], precisions[best_idx],
           s=100, color=PL_PINK, zorder=5, label=f'Umbral óptimo={best_threshold:.2f}')
ax.set_title('Curva Precision-Recall', color=PL_PURPLE, fontweight='bold')
ax.set_xlabel('Recall (Sensibilidad)')
ax.set_ylabel('Precision')
ax.legend(loc='upper right')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# Confusion Matrix (umbral óptimo)
ax = axes[2]
cm = confusion_matrix(y_test, y_pred_opt)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=ax,
            xticklabels=['Pred 0\n(no gol)', 'Pred 1\n(gol)'],
            yticklabels=['Real 0\n(no gol)', 'Real 1\n(gol)'])
ax.set_title(f'Matriz de Confusión\n(umbral óptimo={best_threshold:.2f})',
             color=PL_PURPLE, fontweight='bold')

plt.suptitle(f'Modelo xG V2 — Test Set ({len(y_test):,} disparos)',
             fontsize=14, fontweight='bold', color=PL_PURPLE, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
print('='*64)
print('CLASSIFICATION REPORT — UMBRAL ÓPTIMO')
print('='*64)
print(f'  (Umbral de decisión: P̂(gol) ≥ {best_threshold:.4f})')
print()
print(classification_report(y_test, y_pred_opt,
                             target_names=['No Gol (0)', 'Gol (1)'],
                             zero_division=0))

print()
print('  CLASSIFICATION REPORT — UMBRAL DEFAULT (0.50)')
print()
print(classification_report(y_test, y_pred_default,
                             target_names=['No Gol (0)', 'Gol (1)'],
                             zero_division=0))

---
## SECCIÓN 8: Análisis de Coeficientes — Odds Ratios

Los coeficientes β de la regresión logística representan el **log-odds** — el cambio en el log de las probabilidades ante un incremento de 1 unidad (estandarizada) en la feature. El **Odds Ratio** (OR = exp(β)) es la multiplicación de las probabilidades:

- `OR > 1` → la feature **aumenta** la probabilidad de gol
- `OR < 1` → la feature **reduce** la probabilidad de gol  
- `OR = 1` → la feature no tiene efecto

In [ ]:
# Extraer coeficientes del modelo calibrado
# CalibratedClassifierCV wraps the base estimator — extraemos sus coeficientes promediados
calibrated_clf = calibrated_pipeline.named_steps['calibrated']
scaler_        = calibrated_pipeline.named_steps['scaler']

# Los estimadores calibrados tienen .calibrated_classifiers_
all_betas = np.array([
    est.estimator.coef_[0]
    for est in calibrated_clf.calibrated_classifiers_
])
all_intercepts = np.array([
    est.estimator.intercept_[0]
    for est in calibrated_clf.calibrated_classifiers_
])

betas_mean     = all_betas.mean(axis=0)
betas_std      = all_betas.std(axis=0)
intercept_mean = all_intercepts.mean()
odds_ratios    = np.exp(betas_mean)

df_coef = pd.DataFrame({
    'Feature'        : final_features,
    'β (log-odds)'   : betas_mean,
    'β std (cv)'     : betas_std,
    'OR = exp(β)'    : odds_ratios,
    '|β| magnitud'   : np.abs(betas_mean)
}).sort_values('|β| magnitud', ascending=False)

print('='*72)
print('COEFICIENTES DE LA REGRESIÓN LOGÍSTICA (media de 5 folds calibrados)')
print('='*72)
print(f'  Intercepto β₀: {intercept_mean:+.4f}')
print()
print(f"  {'Feature':<22} {'β':>8}  {'β std':>7}  {'OR=exp(β)':>10}  {'Efecto'}")
print('  ' + '─'*68)

for _, row in df_coef.iterrows():
    bar = '█' * max(1, int(abs(row['β (log-odds)']) * 10))
    direction = 'AUMENTA prob. gol' if row['β (log-odds)'] > 0 else 'REDUCE prob. gol'
    print(f"  {row['Feature']:<22} {row['β (log-odds)']:>+8.4f}  "
          f"{row['β std (cv)']:>7.4f}  {row['OR = exp(β)']:>10.4f}  "
          f"{bar}  ({direction})")

print('  ' + '─'*68)
print()
print('  Lectura: "β std" es la desviación del coeficiente entre folds — valores')
print('  bajos indican coeficientes estables y bien aprendidos.')

In [ ]:
# Ranking visual de magnitudes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = [PL_CYAN if b > 0 else PL_PINK for b in df_coef['β (log-odds)']]

# Odds Ratios
bars = ax1.barh(df_coef['Feature'], df_coef['OR = exp(β)'],
                color=colors, edgecolor='white', linewidth=0.5)
ax1.axvline(1.0, color='white', lw=1.5, linestyle='--', alpha=0.7, label='OR=1 (sin efecto)')
ax1.set_title('Odds Ratios por Feature', color=PL_PURPLE, fontweight='bold')
ax1.set_xlabel('OR = exp(β)')
ax1.invert_yaxis()
for bar, val in zip(bars, df_coef['OR = exp(β)']):
    ax1.text(max(val, 1.0) + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9)
ax1.legend()

# |β| magnitudes con barras de error (β std)
ax2.barh(df_coef['Feature'], df_coef['|β| magnitud'],
         xerr=df_coef['β std (cv)'],
         color=colors, edgecolor='white', linewidth=0.5,
         error_kw={'ecolor': 'white', 'capsize': 4})
ax2.set_title('Magnitud |β| (con varianza entre folds)',
              color=PL_PURPLE, fontweight='bold')
ax2.set_xlabel('|β|')
ax2.invert_yaxis()

cyan_patch = mpatches.Patch(color=PL_CYAN, label='β > 0 (aumenta prob. gol)')
pink_patch = mpatches.Patch(color=PL_PINK, label='β < 0 (reduce prob. gol)')
fig.legend(handles=[cyan_patch, pink_patch], loc='lower center',
           ncol=2, framealpha=0.9, bbox_to_anchor=(0.5, -0.05))

fig.suptitle('Influencia de Cada Feature en P̂(gol)',
             fontsize=14, fontweight='bold', color=PL_PURPLE)
plt.tight_layout()
plt.show()

In [ ]:
print('='*72)
print('NARRATIVA TÁCTICA — INTERPRETACIÓN DE LOS COEFICIENTES')
print('='*72)
print()

# Ordenar para narrativa
df_narrative = df_coef.sort_values('|β| magnitud', ascending=False)

narratives = {
    'dist_al_arco'  : ('β<0 → OR<1 — A mayor distancia, MENOR probabilidad de gol.\n'
                       '    Físicamente intuitivo: disparos desde fuera del área son menos precisos\n'
                       '    y el portero tiene más tiempo de reacción.'),
    'angulo_tiro'   : ('β>0 → OR>1 — A mayor ángulo (apertura hacia portería), MAYOR prob. de gol.\n'
                       '    Ángulo 0 = disparo lateral sin portería enfrente; π/2 = frente al arco.'),
    'goal_mouth_z'  : ('β<0 → OR<1 — A mayor altura de impacto, MENOR prob. de gol.\n'
                       '    Los goles se marcan pegados al piso (media=12.9) vs fallos altos (25.3).\n'
                       '    El portero cubre mejor la altura que la profundidad.'),
    'aim_central'   : ('β<0 → OR<1 — A mayor descentramiento, MENOR prob. de gol.\n'
                       '    aim_central=0 significa disparo al centro exacto de la portería.\n'
                       '    Goles: media=2.6 vs Fallos: media=5.4.'),
    'is_BigChance'  : ('β>0 → OR>1 — Ser Big Chance MULTIPLICA las probabilidades de gol.\n'
                       '    Esta bandera combina varios factores no capturados por las métricas\n'
                       '    continuas: pressing, rebotes, posición de los defensas.'),
    'is_Penalty'    : ('β>0 → OR>1 — Los penales son los disparos con mayor OR del modelo.\n'
                       '    Posición fija, distancia fija (~11m), tasa histórica de conversión ~76%.'),
    'is_OneOnOne'   : ('β>0 → OR>1 — Uno contra uno tiene un OR alto, aunque menos que penalti.\n'
                       '    El jugador enfrenta solo al portero pero la posición es variable.'),
    'threat'        : ('β>0 → OR>1 — Jugadores con mayor amenaza FPL tienen mayor OR.\n'
                       '    Threat es un proxy de la calidad del tirador — destrezas técnicas\n'
                       '    que no capturan las métricas geométricas.'),
}

for i, (_, row) in enumerate(df_narrative.iterrows(), 1):
    feat = row['Feature']
    beta = row['β (log-odds)']
    OR   = row['OR = exp(β)']
    narr = narratives.get(feat, 'Sin narrativa.')
    print(f'  {i}. {feat}')
    print(f'     β={beta:+.4f}  OR={OR:.4f}')
    for line in narr.split('\n'):
        print(f'     {line}')
    print()

In [ ]:
# Ejemplo de Predicción Paso a Paso
print('='*72)
print('EJEMPLO DE PREDICCIÓN PASO A PASO')
print('='*72)

# Elegir un gol real del test set
goal_indices = y_test[y_test == 1].index
miss_indices = y_test[y_test == 0].index

ex_indices = [goal_indices[0], miss_indices[0]]
ex_labels  = ['GOL (clase real = 1)', 'NO GOL (clase real = 0)']

X_test_scaled = scaler_.transform(X_test)
X_test_df     = pd.DataFrame(X_test_scaled, columns=final_features, index=X_test.index)

for ex_idx, ex_label in zip(ex_indices, ex_labels):
    print()
    print(f'  ── Ejemplo: {ex_label} ──')
    ex_raw    = X_test.loc[ex_idx]
    ex_scaled = X_test_df.loc[ex_idx]
    ex_prob   = y_proba[y_test.index.get_loc(ex_idx)]

    print(f'    Jugador: {df.loc[ex_idx, "player_name"] if ex_idx in df.index else "N/A"}')
    print()
    print(f"    {'Feature':<22}  {'Valor raw':>12}  {'Valor escalado':>15}")
    print(f'    {"─"*54}')
    for feat in final_features:
        print(f"    {feat:<22}  {ex_raw[feat]:>12.4f}  {ex_scaled[feat]:>15.4f}")
    print()
    print(f'    P̂(gol) = {ex_prob:.4f}  →  Predicción: {"GOL" if ex_prob >= best_threshold else "No gol"} '
          f'(umbral={best_threshold:.3f})')

---
## SECCIÓN 9: Efectos de Jugador — Variable `player_name` como Efecto Fijo

### Motivación

El modelo V2 captura la **geometría del disparo** con alta fidelidad (ROC-AUC 0.836),  
pero ignora un factor fundamental: **¿quién dispara?** No es lo mismo un remate de  
Erling Haaland desde 16m que el mismo tiro ejecutado por un lateral derecho.

Los modelos xG profesionales (StatsBomb, Opta) capturan esto mediante **efectos  
aleatorios por jugador** (random intercepts). Aquí implementamos la versión de efectos  
fijos con regularización L2, que es técnicamente equivalente para un conjunto cerrado  
de jugadores.

### ¿Qué mide `β_jugador`?

> El efecto de jugador es el **residuo sobre el modelo geométrico base**:  
> cuánto convierte un jugador **por encima o por debajo** de lo que predice  
> la distancia, ángulo y contexto del disparo.

$$\text{logit}(P(\text{gol})) = \text{INTERCEPT} + \beta_{\text{dist}} \cdot d + \beta_{\text{ang}} \cdot \alpha + \ldots + \beta_{\text{jugador}}$$

Un `β_jugador > 0` significa que el jugador convierte **más** de lo que dicta la geometría  
(mejor técnica de definición, remate más potente o más preciso).  
Un `β_jugador < 0` significa que convierte **menos** de lo esperado para esa posición.

### Umbral de datos mínimos

Solo estimamos el efecto para jugadores con **≥ 30 disparos** en el dataset.  
Con menos tiros, el coeficiente es demasiado ruidoso para ser confiable.


In [ ]:
# ── Sección 9.1: Inventario de tiros por jugador ─────────────────────────
MIN_SHOTS = 30

df['is_goal'] = df['is_goal'].astype(int)

player_stats = (
    df.groupby('player_name')
      .agg(tiros=('is_goal', 'count'),
           goles=('is_goal', 'sum'))
      .reset_index()
)
player_stats['conv_obs'] = (player_stats['goles'] / player_stats['tiros']).round(4)
player_stats = player_stats.sort_values('tiros', ascending=False)

eligible = player_stats[player_stats['tiros'] >= MIN_SHOTS]

print('=' * 65)
print(f'INVENTARIO DE JUGADORES — {MIN_SHOTS}+ TIROS')
print('=' * 65)
print(f'  Total jugadores en dataset  : {len(player_stats)}')
print(f'  Jugadores con ≥{MIN_SHOTS} tiros     : {len(eligible)}')
print(f'  Tiros cubiertos             : {eligible["tiros"].sum():,} / {len(df):,} ({eligible["tiros"].sum()/len(df)*100:.1f}%)')
print(f'  Conv. global                : {df["is_goal"].mean():.4f} ({df["is_goal"].mean()*100:.2f}%)')
print()
print(f"  {'Jugador':<30} {'Tiros':>6} {'Goles':>6} {'Conv':>7}")
print('  ' + '-'*54)
for _, row in eligible.iterrows():
    bar = '█' * int(row['conv_obs'] * 30)
    print(f"  {row['player_name']:<30} {int(row['tiros']):>6} {int(row['goles']):>6} {row['conv_obs']:>7.3f}  {bar}")


In [ ]:
# ── Sección 9.2: Modelo con Efectos Fijos de Jugador (L2) ────────────────
#
# Estrategia: añadir dummies de jugador al modelo logístico con regularización
# L2 (C=0.5). La regularización aplica shrinkage — coeficientes de jugadores
# con pocos tiros se contraen hacia 0 (hacia el promedio), reduciendo el ruido.
#
# C=0.5 es una regularización moderada: más fuerte que C=1 (default) para
# controlar la varianza de los efectos de jugador, sin colapsar los efectos
# reales de los grandes finalizadores.

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import pandas as pd, numpy as np

MIN_SHOTS = 30
eligible_players = player_stats[player_stats['tiros'] >= MIN_SHOTS]['player_name'].tolist()

df_top = df[df['player_name'].isin(eligible_players)].copy()
print(f'Filas en subconjunto de jugadores elegibles: {len(df_top):,}')

base_feats = ['dist_al_arco', 'angulo_tiro', 'goal_mouth_z', 'aim_central',
              'is_BigChance', 'is_Penalty', 'is_OneOnOne']

# Dummies de jugador (drop_first para evitar multicolinealidad perfecta)
player_dummies = pd.get_dummies(df_top['player_name'], prefix='p', drop_first=True)
ref_player = [p for p in eligible_players if ('p_' + p) not in player_dummies.columns][0]
print(f'Jugador de referencia (β=0): {ref_player}')

X_full = pd.concat([
    df_top[base_feats].reset_index(drop=True),
    player_dummies.reset_index(drop=True)
], axis=1)
y_full = df_top['is_goal'].values

# Escalar solo las features base (los dummies 0/1 no necesitan escala)
scaler_p = StandardScaler()
X_full_values = X_full.values.copy().astype(float)
X_full_values[:, :len(base_feats)] = scaler_p.fit_transform(X_full_values[:, :len(base_feats)])

# Ajustar modelo con regularización L2 moderada
lr_player = LogisticRegression(max_iter=2000, C=0.5, solver='lbfgs')
lr_player.fit(X_full_values, y_full)

print(f'\nConvergencia: {lr_player.n_iter_[0]} iteraciones')
print(f'Coeficientes totales: {len(lr_player.coef_[0])} ({len(base_feats)} base + {len(player_dummies.columns)} jugadores)')


In [ ]:
# ── Sección 9.3: Extracción de β_jugador y Ranking ───────────────────────

feature_names = list(X_full.columns)
coef_dict = dict(zip(feature_names, lr_player.coef_[0]))

# Separar coefs base de coefs jugador
base_coefs = {k: v for k, v in coef_dict.items() if not k.startswith('p_')}
player_raw = {k.replace('p_', ''): v for k, v in coef_dict.items() if k.startswith('p_')}
player_raw[ref_player] = 0.0  # jugador de referencia

# Calcular tasa de conversión predicha por modelo base (sin efecto jugador)
df_top2 = df_top.copy()
X_base_scaled = scaler_p.transform(df_top2[base_feats])
lr_base_only = LogisticRegression(max_iter=1000, C=1.0)
lr_base_only.fit(X_base_scaled, df_top2['is_goal'].values)
df_top2['p_base'] = lr_base_only.predict_proba(X_base_scaled)[:, 1]

player_summary = player_stats[player_stats['player_name'].isin(eligible_players)].copy()
pred_rates = df_top2.groupby('player_name')['p_base'].mean().reset_index()
pred_rates.columns = ['player_name', 'conv_pred_base']
player_summary = player_summary.merge(pred_rates, on='player_name')
player_summary['beta_jugador'] = player_summary['player_name'].map(player_raw).round(3)
player_summary['conv_adj'] = (1 / (1 + np.exp(-(
    np.log(player_summary['conv_pred_base'] / (1 - player_summary['conv_pred_base']))
    + player_summary['beta_jugador']
)))).round(3)
player_summary = player_summary.sort_values('beta_jugador', ascending=False)

print('=' * 80)
print('EFECTOS DE JUGADOR — Regresión Logística L2 (C=0.5, ≥30 tiros)')
print('=' * 80)
print(f"  {'Jugador':<30} {'Tiros':>6} {'Conv.Obs':>9} {'Conv.Pred':>10} {'β_jugador':>10} {'Conv.Adj':>9}")
print('  ' + '-'*76)
for _, row in player_summary.iterrows():
    sign = '+' if row['beta_jugador'] >= 0 else ''
    flag = ' ↑' if row['beta_jugador'] > 0.2 else (' ↓' if row['beta_jugador'] < -0.2 else '')
    print(f"  {row['player_name']:<30} {int(row['tiros']):>6} "          f"{row['conv_obs']:>9.3f} {row['conv_pred_base']:>10.3f} "          f"{sign}{row['beta_jugador']:>9.3f} {row['conv_adj']:>9.3f}{flag}")


In [ ]:
# ── Sección 9.4: Visualización de Efectos de Jugador ─────────────────────

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

PL_PURPLE = '#37003c'
PL_CYAN   = '#00ff85'
PL_WHITE  = '#f0f0f0'

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 9))
fig.patch.set_facecolor(PL_PURPLE)

# ── Panel 1: Ranking de efectos ──────────────────────────────────────────
top_n = 20  # top 10 + bottom 10
top10 = player_summary.head(10)
bot10 = player_summary.tail(10)
display = pd.concat([top10, bot10]).drop_duplicates()

names = display['player_name'].str.split().str[-1]  # solo apellido
betas = display['beta_jugador']
colors = [PL_CYAN if b >= 0 else '#ff4d6d' for b in betas]

bars = ax1.barh(range(len(display)), betas, color=colors, edgecolor='none', height=0.7)
ax1.set_yticks(range(len(display)))
ax1.set_yticklabels(names, color=PL_WHITE, fontsize=9)
ax1.axvline(0, color='white', linewidth=0.6, alpha=0.4)
ax1.set_facecolor(PL_PURPLE)
ax1.tick_params(colors=PL_WHITE, labelsize=8)
ax1.set_xlabel('β_jugador (log-odds sobre media)', color=PL_WHITE, fontsize=9)
ax1.set_title('Top 10 / Bottom 10\nEfectos de Jugador', color=PL_WHITE, fontsize=11, fontweight='bold')
for spine in ax1.spines.values():
    spine.set_edgecolor('rgba(255,255,255,0.1)')
    spine.set_alpha(0.15)

# Anotar valores
for i, (bar, beta) in enumerate(zip(bars, betas)):
    ax1.text(beta + (0.02 if beta >= 0 else -0.02), i,
             f'{beta:+.2f}', va='center',
             ha='left' if beta >= 0 else 'right',
             color=PL_WHITE, fontsize=7.5)

# ── Panel 2: Scatter conv. observada vs predicha base ───────────────────
ax2.set_facecolor(PL_PURPLE)
sc = ax2.scatter(
    player_summary['conv_pred_base'],
    player_summary['conv_obs'],
    c=player_summary['beta_jugador'],
    cmap='RdYlGn', vmin=-0.6, vmax=0.6,
    s=player_summary['tiros'] * 1.2,
    alpha=0.85, edgecolors='white', linewidth=0.4
)
# Línea y=x (conv. obs = conv. pred → efecto neutro)
lim = max(player_summary['conv_pred_base'].max(), player_summary['conv_obs'].max()) + 0.02
ax2.plot([0, lim], [0, lim], '--', color='white', alpha=0.35, linewidth=1)

# Etiquetas jugadores con mayor efecto
for _, row in player_summary[abs(player_summary['beta_jugador']) > 0.3].iterrows():
    ax2.annotate(row['player_name'].split()[-1],
                 (row['conv_pred_base'], row['conv_obs']),
                 fontsize=7, color=PL_WHITE, alpha=0.85,
                 xytext=(4, 4), textcoords='offset points')

cbar = fig.colorbar(sc, ax=ax2)
cbar.set_label('β_jugador', color=PL_WHITE, fontsize=9)
cbar.ax.yaxis.set_tick_params(color=PL_WHITE)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=PL_WHITE)

ax2.set_xlabel('Conversión predicha (modelo base)', color=PL_WHITE, fontsize=9)
ax2.set_ylabel('Conversión observada real', color=PL_WHITE, fontsize=9)
ax2.set_title('Obs. vs Predicha por Modelo Base\n(tamaño = nº tiros, color = β_jugador)', 
              color=PL_WHITE, fontsize=11, fontweight='bold')
ax2.tick_params(colors=PL_WHITE, labelsize=8)
for spine in ax2.spines.values():
    spine.set_alpha(0.15)

fig.suptitle('Sección 9: Efectos de Jugador — Premier League 24/25', 
             color=PL_WHITE, fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../../img/player_effects_xg.png', dpi=150, bbox_inches='tight',
            facecolor=PL_PURPLE)
plt.show()
print('Gráfica guardada en img/player_effects_xg.png')


In [ ]:
# ── Sección 9.5: Coeficientes base del modelo ampliado ───────────────────

print('=' * 65)
print('COEFICIENTES BASE — Modelo con efectos de jugador (L2 C=0.5)')
print('=' * 65)
print(f"  {'Feature':<22} {'β':>8} {'OR=exp(β)':>12} {'Efecto'}  ")
print('  ' + '-'*60)
for feat in base_feats:
    b = base_coefs[feat]
    OR = np.exp(b)
    direction = '↑ aumenta prob. gol' if b > 0 else '↓ reduce prob. gol'
    print(f"  {feat:<22} {b:>8.4f} {OR:>12.4f}   {direction}")

print()
print('Interpretación:')
print('  Los coeficientes base son ligeramente menores en magnitud')
print('  respecto al modelo sin jugador porque parte de la variabilidad')
print('  ya es absorbida por los efectos individuales.')
print()
print('Casos notables (β_jugador):')
notable = player_summary[abs(player_summary['beta_jugador']) > 0.3][
    ['player_name','tiros','conv_obs','conv_pred_base','beta_jugador']
].sort_values('beta_jugador', ascending=False)
for _, row in notable.iterrows():
    sign = '+' if row['beta_jugador'] > 0 else ''
    print(f"  {row['player_name']:<28} β={sign}{row['beta_jugador']:.3f}  "          f"(obs={row['conv_obs']:.3f}, pred_base={row['conv_pred_base']:.3f})")

print()
print('Nota: Mohamed Salah β=-0.287 no significa que sea mal finalizador.')
print('      En 24/25 tiró frecuentemente desde ángulos difíciles — el modelo')
print('      base ya captura esas posiciones como de baja conversión, y Salah')
print('      no superó esa predicción geométrica en esta temporada.')
print()
print('Nota: Haaland β=+0.115 es bajo porque su geometría era tan buena')
print('      que el modelo base ya lo predecía bien — residuo pequeño.')


In [ ]:
# ── Sección 9.6: Exportar coeficientes calibrados al dashboard ──────────
#
# Los coeficientes del modelo completo (con scaler en unidades estándar) no
# pueden usarse directamente en JavaScript porque el dashboard trabaja en
# unidades brutas (metros, radianes). Aquí documentamos los coeficientes
# calibrados que se usaron en el dashboard junto con su procedencia.

DASHBOARD_COEFS = {
    'INTERCEPT' : -2.635,
    'dist'      : -0.0175,   # β por metro de distancia (unidades brutas)
    'angle'     : +0.088,    # β por grado de ángulo  
    'gmz'       : -0.040,    # β por unidad de altura (escala 0-50)
    'aim'       : -0.050,    # β por unidad de centralidad (escala 0-50)
    'bigchance' : +1.530,    # OR ≈ 4.62 — ocasión clara
    'penalty'   : +0.800,    # bono adicional sobre BigChance
    'oneone'    :  -0.100,   # mano a mano (efecto negativo residual en datos)
}

PLAYER_EFFECTS_DASHBOARD = {
    # Extraídos de lr_player.coef_ — ya en escala natural (sin estandarizar dummies)
    player: round(player_raw[player], 3)
    for player in sorted(player_raw.keys(), key=lambda p: -player_raw[p])
}

print('PLAYER_EFFECTS = {')
for p, v in PLAYER_EFFECTS_DASHBOARD.items():
    sign = '+' if v >= 0 else ''
    print(f"    '{p}': {sign}{v},")
print('}')
print()
print(f'Total jugadores exportados: {len(PLAYER_EFFECTS_DASHBOARD)}')
print()
print('Validaciones rápidas (dashboard demo):')
import math
def xg_demo(dist, angle_deg, gmz, aim, bigchance, penalty, oneone, player='__avg__'):
    b = DASHBOARD_COEFS
    bp = PLAYER_EFFECTS_DASHBOARD.get(player, 0)
    logit = (b['INTERCEPT'] + b['dist']*dist + b['angle']*angle_deg
             + b['gmz']*gmz + b['aim']*aim
             + b['bigchance']*bigchance + b['penalty']*penalty
             + b['oneone']*oneone + bp)
    return 1 / (1 + math.exp(-logit))

cases = [
    ('Penalti — promedio',       xg_demo(11.5, 25, 12, 4, 1, 1, 0)),
    ('Penalti — Haaland',        xg_demo(11.5, 25, 12, 4, 1, 1, 0, 'Erling Haaland')),
    ('Penalti — David Brooks',   xg_demo(11.5, 25, 12, 4, 1, 1, 0, 'David Brooks')),
    ('6 yardas BigChance',       xg_demo(5.5,  34, 8,  2, 1, 0, 0)),
    ('10m BigChance',            xg_demo(10,   22, 12, 5, 1, 0, 0)),
    ('20m sin contexto',         xg_demo(20,   14, 12, 8, 0, 0, 0)),
    ('25m esquina ángulo cerrado', xg_demo(25, 8, 30, 35, 0, 0, 0)),
]
for name, xg_val in cases:
    bar = '█' * int(xg_val * 40)
    print(f'  {name:<35} xG = {xg_val:.3f}  {bar}')


### Conclusiones de la Sección 9

| Hallazgo | Detalle |
|:---|:---|
| **Jugadores elegibles** | 70 jugadores con ≥30 tiros |
| **Tiros cubiertos** | ~85% del dataset en los 70 jugadores |
| **Jugador de referencia (β=0)** | Determinado por `drop_first` en dummies |
| **Regularización** | L2 con C=0.5 — shrinkage hacia 0 para jugadores con pocos tiros |
| **Salah (β=−0.287)** | No es mal finalizador — en 24/25 tiró mucho desde posiciones difíciles |
| **Haaland (β=+0.115)** | Efecto residual pequeño porque su geometría ya era excelente |
| **Bruno Guimarães (β=+0.706)** | Convierte significativamente más de lo que predice su posición media |

### Limitación importante

El efecto de jugador estimado aquí refleja **una sola temporada** (24/25).  
Un jugador que cambió de rol, tuvo lesiones o atravesó una racha particular  
puede tener un β sesgado. En producción, se usaría un modelo de efectos  
aleatorios con ventana deslizante de 2-3 temporadas.

### Implementación en el Dashboard

El coeficiente `β_jugador` se suma directamente al logit del modelo base,  
exactamente como cualquier otro coeficiente:

```python
logit = INTERCEPT + β_dist·d + β_ang·α + β_BC·BC + β_pen·pen + β_OO·oo + β_jugador
xG = sigmoid(logit)
```

El dashboard incluye 27 jugadores seleccionados de los 70 elegibles,  
agrupados por rol con su efecto visible al usuario en tiempo real.


---
## RESUMEN EJECUTIVO

| Métrica | Valor |
|---|---|
| **ROC-AUC (Test)** | Ver output |
| **PR-AUC (Test)** | Ver output |
| Features finales | Ver output |
| Features eliminadas VIF | Ver output |
| Features eliminadas Spearman | Ver output |
| Features nuevas añadidas | `goal_mouth_z`, `aim_central` |
| Calibración | Platt scaling (CalibratedClassifierCV) |
| Validación cruzada | StratifiedKFold(k=5) |
| Desbalance tratado | `class_weight='balanced'` |

**Conclusión:** El Modelo xG V2 incorpora la dimensión de altura del disparo (`goal_mouth_z`) y la centralidad lateral (`aim_central`), añadiendo información 3D sobre el destino del balón que la V1 ignoraba completamente. La calibración Platt corrige las probabilidades sobreestimadas por el `class_weight='balanced'`, produciendo scores xG más interpretables y utilizables en contextos de scouting.